In [1]:
import numpy as np
import pandas as pd

df=pd.read_csv(r"dataset_clap.csv")
x=np.array(df)
print(x.shape)

(1500, 155)


In [2]:
x.min(), x.max()

(-0.7561202049255371, 357.68901303330284)

In [3]:
x

array([[ 4.32098806e-02, -6.38195932e-01, -2.34563038e-01, ...,
         2.01015719e+01,  0.00000000e+00,  3.26755763e+01],
       [ 4.33488339e-02, -6.52900636e-01, -1.93292826e-01, ...,
         2.18348477e+01,  0.00000000e+00,  4.09634329e+01],
       [ 4.34381925e-02, -6.54311895e-01, -1.66693702e-01, ...,
         2.21959375e+01,  0.00000000e+00,  4.48085526e+01],
       ...,
       [ 1.97095480e-02, -6.16171896e-01, -1.60277620e-01, ...,
         2.65732326e+00,  0.00000000e+00,  1.33580739e+02],
       [ 2.00743992e-02, -6.10753298e-01, -1.70257404e-01, ...,
         1.51557352e+01,  0.00000000e+00,  1.74845244e+02],
       [ 1.79990418e-02, -6.08126640e-01, -2.05310717e-01, ...,
         2.25106712e+01,  0.00000000e+00,  1.54466080e+02]])

In [7]:
import pandas as pd
import numpy as np
from imitation.data.types import Trajectory

def normalize_landmarks(obs):
    # obs shape: (N, 132). Landmarks=33, each has (x,y,z,v) = 4 units.
    # Hip indices: 23 (right hip), 24 (left hip)
    r_hip_idx, l_hip_idx = 23 * 4, 24 * 4
    # Shoulder indices: 12 (right), 11 (left)
    r_sho_idx, l_sho_idx = 12 * 4, 11 * 4
    
    # Calculate mid-hip
    mid_hip_x = (obs[:, r_hip_idx] + obs[:, l_hip_idx]) / 2.0
    mid_hip_y = (obs[:, r_hip_idx+1] + obs[:, l_hip_idx+1]) / 2.0
    mid_hip_z = (obs[:, r_hip_idx+2] + obs[:, l_hip_idx+2]) / 2.0
    
    # Calculate mid-shoulder
    mid_sho_x = (obs[:, r_sho_idx] + obs[:, l_sho_idx]) / 2.0
    mid_sho_y = (obs[:, r_sho_idx+1] + obs[:, l_sho_idx+1]) / 2.0
    mid_sho_z = (obs[:, r_sho_idx+2] + obs[:, l_sho_idx+2]) / 2.0
    
    # Calculate torso scaled height
    torso_height = np.sqrt(
        (mid_sho_x - mid_hip_x)**2 + 
        (mid_sho_y - mid_hip_y)**2 + 
        (mid_sho_z - mid_hip_z)**2
    )
    
    # Prevent div by 0 just in case
    torso_height = np.where(torso_height == 0, 1.0, torso_height)
    
    norm_obs = np.copy(obs)
    for i in range(33):
        idx = i * 4
        # Center at mid-hip, and scale by torso_height
        norm_obs[:, idx]   = (obs[:, idx] - mid_hip_x) / torso_height     # x
        norm_obs[:, idx+1] = (obs[:, idx+1] - mid_hip_y) / torso_height   # y
        norm_obs[:, idx+2] = (obs[:, idx+2] - mid_hip_z) / torso_height   # z
        # visibility (idx+3) remains essentially unchanged
        
    return norm_obs

def load_expert_data(file_path):
    df = pd.read_csv(file_path)
    data = df.values
    
    # Split Observations and Actions
    obs = data[:, :132]  
    acts = data[:, 132:]
    
    # Normalize Landmarks
    obs = normalize_landmarks(obs)
    
    # Normalize Joints to [-1, 1]
    acts = (acts - 180.0) / 180.0 
    
    # Cast safely to float32 preferred by Torch
    obs = obs.astype(np.float32)
    acts = acts.astype(np.float32)
    
    # GAIL needs terminal=True observation to match state sequence S_{0...T} for actions A_{0...T-1}
    obs_with_terminal = np.vstack([obs, obs[-1]])
    
    return Trajectory(obs=obs_with_terminal, acts=acts, infos=None, terminal=True)

expert_trajs = [
    load_expert_data("dataset_clap.csv"),
    load_expert_data("dataset_disco.csv"),
    load_expert_data("dataset_hello.csv"),
    load_expert_data("dataset_wakanda.csv"),
    load_expert_data("dataset_zombie.csv")
]

print(f"✅ Loaded {len(expert_trajs)} Expert Trajectories.")
print(f"Observation Shape: {expert_trajs[0].obs.shape}")
print(f"Action Shape: {expert_trajs[0].acts.shape}")

✅ Loaded 5 Expert Trajectories.
Observation Shape: (1501, 132)
Action Shape: (1500, 23)


In [8]:
import pickle

# Serialize trajectories into a pickle file so stable-baselines/GAIL can load it anytime
with open("normalized_expert_trajectories.pkl", "wb") as f:
    pickle.dump(expert_trajs, f)
print("Saved expert trajectories to normalized_expert_trajectories.pkl")

Saved expert trajectories to normalized_expert_trajectories.pkl


In [10]:
!python train_lstm.py

Using device: cpu
Loading dataset...
Dataset successfully loaded. Length: 7355
